In [ ]:
import pandas as pd
import numpy as np

In [ ]:
df = pd.read_csv('../../data/raw/wb_income_classification.csv', encoding='latin-1', sep=';')

Step 1: Use exploratory function

In [ ]:
def analyze_dataframe(df):
    """
    Comprehensive DataFrame analysis showing missing values, unique values, and value counts
    """
    
    print("=" * 80)
    print("DATAFRAME ANALYSIS")
    print("=" * 80)
    
    # Section 1: Basic Info
    print("\n1. BASIC DATAFRAME INFORMATION")
    print("-" * 40)
    print(f"Shape: {df.shape[0]} rows × {df.shape[1]} columns")
    print(f"Total cells: {df.size}")
    print(f"Memory usage: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
    
    # Section 2: Missing Values Analysis
    print("\n2. MISSING VALUES BY COLUMN")
    print("-" * 40)
    
    # Create a DataFrame with missing value info
    missing_info = pd.DataFrame({
        'Column': df.columns,
        'Data Type': [df[col].dtype for col in df.columns],
        'Non-Null Count': [df[col].count() for col in df.columns],
        'Missing Count': [df[col].isnull().sum() for col in df.columns],
        'Missing %': [round(df[col].isnull().sum() / len(df) * 100, 2) for col in df.columns]
    })
    
    # Sort by missing count descending
    missing_info = missing_info.sort_values('Missing Count', ascending=False)
    
    # Format the output
    for _, row in missing_info.iterrows():
        missing_bar = '█' * int(row['Missing %'] / 5)  # Visual bar for missing percentage
        print(f"{row['Column'][:30]:<30} "
              f"| Type: {str(row['Data Type']):<10} "
              f"| Missing: {int(row['Missing Count']):>6} "
              f"| {row['Missing %']:>5}% {missing_bar}")
    
    # Section 3: Unique Values Analysis (for object/category columns)
    print("\n3. UNIQUE VALUES IN CATEGORICAL COLUMNS")
    print("-" * 40)
    
    categorical_cols = df.select_dtypes(include=['object', 'category']).columns
    
    if len(categorical_cols) > 0:
        unique_info = pd.DataFrame({
            'Column': categorical_cols,
            'Unique Values': [df[col].nunique() for col in categorical_cols],
            'Sample Values': [str(list(df[col].dropna().unique()[:5])) for col in categorical_cols]
        })
        unique_info = unique_info.sort_values('Unique Values', ascending=False)
        
        for _, row in unique_info.iterrows():
            print(f"{row['Column'][:30]:<30} "
                  f"| Unique: {row['Unique Values']:<5} "
                  f"| Samples: {row['Sample Values'][:50]}")
    else:
        print("No categorical columns found")
    
    # Section 4: Numeric Columns Statistics
    print("\n4. NUMERIC COLUMNS SUMMARY")
    print("-" * 40)
    
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    
    if len(numeric_cols) > 0:
        numeric_stats = df[numeric_cols].describe().T
        numeric_stats['missing'] = df[numeric_cols].isnull().sum()
        print(numeric_stats[['count', 'missing', 'mean', 'min', 'max']])
    else:
        print("No numeric columns found")
    
    return missing_info

def show_value_counts(df, column_name, top_n=10):
    """
    Display value counts for a specific column with formatting
    """
    print("\n" + "=" * 80)
    print(f"VALUE COUNTS FOR COLUMN: '{column_name}'")
    print("=" * 80)
    
    if column_name not in df.columns:
        print(f"Error: Column '{column_name}' not found in DataFrame")
        return
    
    # Get value counts
    counts = df[column_name].value_counts()
    
    # Get value counts with percentages
    counts_pct = df[column_name].value_counts(normalize=True) * 100
    
    # Create a combined DataFrame
    value_counts_df = pd.DataFrame({
        'Count': counts,
        'Percentage': counts_pct,
        'Cumulative %': counts_pct.cumsum()
    })
    
    # Show top N or all if less than top_n
    display_count = min(top_n, len(counts))
    
    print(f"\nTop {display_count} values (out of {len(counts)} unique values):")
    print("-" * 60)
    
    for i, (value, count) in enumerate(counts.head(display_count).items(), 1):
        percentage = counts_pct[value]
        bar = '█' * int(percentage / 2)  # Visual bar (max 50 chars)
        
        # Truncate value if too long
        value_str = str(value)[:30]
        
        print(f"{i:2}. {value_str:<30} "
              f"| Count: {count:<8} "
              f"| {percentage:>5.1f}% {bar}")
    
    # Show missing values if any
    missing_count = df[column_name].isnull().sum()
    if missing_count > 0:
        missing_pct = (missing_count / len(df)) * 100
        print(f"\nMissing values: {missing_count} ({missing_pct:.1f}%)")
    
    return value_counts_df

In [ ]:
missing_info = analyze_dataframe(df)

DATAFRAME ANALYSIS

1. BASIC DATAFRAME INFORMATION
----------------------------------------
Shape: 224 rows × 40 columns
Total cells: 8960
Memory usage: 0.08 MB

2. MISSING VALUES BY COLUMN
----------------------------------------
1987                           | Type: str        | Missing:     58 | 25.89% █████
1988                           | Type: str        | Missing:     57 | 25.45% █████
1989                           | Type: str        | Missing:     54 | 24.11% ████
1990                           | Type: str        | Missing:     44 | 19.64% ███
1991                           | Type: str        | Missing:     27 | 12.05% ██
1992                           | Type: str        | Missing:     20 |  8.93% █
1993                           | Type: str        | Missing:     19 |  8.48% █
1994                           | Type: str        | Missing:     18 |  8.04% █
1995                           | Type: str        | Missing:     18 |  8.04% █
1998                           | Type: str  

C:\Users\vanes\AppData\Local\Temp\ipykernel_26764\315415878.py:48: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = df.select_dtypes(include=['object', 'category']).columns


Step 2: Replace '..' with pandas na, as they are missing values

In [ ]:
df = df.replace('..', pd.NA)

In [4]:
df.head()

,country_code,country_name,1987,1988,1989,1990,1991,1992,1993,1994,...,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024
0,AFG,Afghanistan,L,L,L,L,L,L,L,L,...,L,L,L,L,L,L,L,L,L,L
1,ALB,Albania,NaN,NaN,NaN,LM,LM,LM,L,L,...,UM,UM,UM,UM,UM,UM,UM,UM,UM,UM
2,DZA,Algeria,UM,UM,LM,LM,LM,LM,LM,LM,...,UM,UM,UM,UM,LM,LM,LM,LM,UM,UM
3,ASM,American Samoa,H,H,H,UM,UM,UM,UM,UM,...,UM,UM,UM,UM,UM,UM,UM,H,H,H
4,AND,Andorra,NaN,NaN,NaN,H,H,H,H,H,...,H,H,H,H,H,H,H,H,H,H


Step 3: Melt dataframe

Explanation: At first, each year was a separate column. But we want each year to be a separate row.

In [5]:
# Melt the dataframe: convert years from columns to rows
df_melted = pd.melt(
    df, 
    id_vars=['country_code', 'country_name'],  # Keep these as identifiers
    var_name='year',                            # Column names become year values
    value_name='classification'                 # The values become classification
)

# Sort for better readability (optional)
df_melted = df_melted.sort_values(['country_name', 'year']).reset_index(drop=True)

# Check the result
print(df_melted.shape)  # (8512, 4)
print(df_melted.head())

(8512, 4)
  country_code country_name  year classification
0          AFG  Afghanistan  1987              L
1          AFG  Afghanistan  1988              L
2          AFG  Afghanistan  1989              L
3          AFG  Afghanistan  1990              L
4          AFG  Afghanistan  1991              L


Check

In [7]:
analyze_dataframe(df_melted)

DATAFRAME ANALYSIS

1. BASIC DATAFRAME INFORMATION
----------------------------------------
Shape: 8512 rows × 4 columns
Total cells: 34048
Memory usage: 0.41 MB

2. MISSING VALUES BY COLUMN
----------------------------------------
classification                 | Type: str        | Missing:    635 |  7.46% █
country_code                   | Type: str        | Missing:      0 |   0.0% 
country_name                   | Type: str        | Missing:      0 |   0.0% 
year                           | Type: str        | Missing:      0 |   0.0% 

3. UNIQUE VALUES IN CATEGORICAL COLUMNS
----------------------------------------
country_code                   | Unique: 224   | Samples: ['AFG', 'ALB', 'DZA', 'ASM', 'AND']
country_name                   | Unique: 224   | Samples: ['Afghanistan', 'Albania', 'Algeria', 'American Sa
year                           | Unique: 38    | Samples: ['1987', '1988', '1989', '1990', '1991']
classification                 | Unique: 5     | Samples: ['L', 'LM', '

C:\Users\vanes\AppData\Local\Temp\ipykernel_26764\315415878.py:48: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = df.select_dtypes(include=['object', 'category']).columns


,Column,Data Type,Non-Null Count,Missing Count,Missing %
3,classification,str,7877,635,7.46
0,country_code,str,8512,0,0.00
1,country_name,str,8512,0,0.00
2,year,str,8512,0,0.00


I see 'LM*' as a unique value in 'classification'

This is what the codebook says about that special value:

"* At this time, there were Yemen, PDR (L) and Yemen, Arab Rep. (LM); combined they would have been LM."

Let's check which rows have that classification

In [ ]:
mask = df_melted['classification'] == 'LM*'
print(df_melted[mask])

     country_code country_name  year classification
8360          YEM  Yemen, Rep.  1987            LM*
8361          YEM  Yemen, Rep.  1988            LM*


Only two instances. 

As the asterisk only was for explanation for why the classification is noted as LM. We can safely remove the asterisk.

Step 4: Remove asterisks making 'LM*' into 'LM'

In [11]:
# Remove all asterisks from the column
df_melted['classification'] = df_melted['classification'].str.replace('*', '', regex=False)

Check

In [12]:
analyze_dataframe(df_melted)

DATAFRAME ANALYSIS

1. BASIC DATAFRAME INFORMATION
----------------------------------------
Shape: 8512 rows × 4 columns
Total cells: 34048
Memory usage: 0.41 MB

2. MISSING VALUES BY COLUMN
----------------------------------------
classification                 | Type: str        | Missing:    635 |  7.46% █
country_code                   | Type: str        | Missing:      0 |   0.0% 
country_name                   | Type: str        | Missing:      0 |   0.0% 
year                           | Type: str        | Missing:      0 |   0.0% 

3. UNIQUE VALUES IN CATEGORICAL COLUMNS
----------------------------------------
country_code                   | Unique: 224   | Samples: ['AFG', 'ALB', 'DZA', 'ASM', 'AND']
country_name                   | Unique: 224   | Samples: ['Afghanistan', 'Albania', 'Algeria', 'American Sa
year                           | Unique: 38    | Samples: ['1987', '1988', '1989', '1990', '1991']
classification                 | Unique: 4     | Samples: ['L', 'LM', '

C:\Users\vanes\AppData\Local\Temp\ipykernel_26764\315415878.py:48: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = df.select_dtypes(include=['object', 'category']).columns


,Column,Data Type,Non-Null Count,Missing Count,Missing %
3,classification,str,7877,635,7.46
0,country_code,str,8512,0,0.00
1,country_name,str,8512,0,0.00
2,year,str,8512,0,0.00


Ok good

Step 5: Save it to csv

In [13]:
df_melted.to_csv('../../data/processed/unmapped_wb_income_classification.csv', index=False)

### To do for this dataset (4/4/2026 20:00)

- Fix missingness
- Map country names to iso3 codes